**Histone modifier loss cohort construction**

This notebook identifies TCGA tumours with loss-of-function mutations or candidate homozygous deletions affecting selected histone modifier genes. The output cohort is used for downstream TE differential expression analysis.

Genes included: KDM6A, KDM6B, NSD1, NSD2 and SETD2.

In [ ]:
# Load packages

library(TCGAbiolinks)
library(dplyr)
library(readr)
library(stringr)
library(tibble)
library(purrr)
library(tidyr)

In [ ]:
# Analysis settings

project_root <- "/home/kennes38/ResearchProject"

if (dir.exists(project_root)) {
  setwd(project_root)
}

gene_set_label <- "histone_modifier_loss"

target_genes <- c("KDM6A", "KDM6B", "NSD1", "NSD2", "SETD2")

output_dir <- file.path(getwd(), "12_histone_modifier_LoFHomDel_finalcohort")
gdc_dir <- file.path(output_dir, "GDCdata")

dir.create(output_dir, recursive = TRUE, showWarnings = FALSE)
dir.create(gdc_dir, recursive = TRUE, showWarnings = FALSE)

# Use "test" first, then change to "full" when ready.
run_mode <- "test"

homdel_threshold <- -1.5

lof_classes <- c(
  "Frame_Shift_Del",
  "Frame_Shift_Ins",
  "Nonsense_Mutation",
  "Splice_Site",
  "Translation_Start_Site",
  "Nonstop_Mutation"
)

all_tcga_projects <- TCGAbiolinks::getGDCprojects() %>%
  filter(str_detect(project_id, "^TCGA-")) %>%
  pull(project_id) %>%
  sort()

projects_to_run <- if (run_mode == "test") {
  head(all_tcga_projects, 2)
} else {
  all_tcga_projects
}

projects_to_run

In [ ]:
# Set Gene Co-ordinates

# GRCh38 coordinates used to overlap genes with copy-number segments.
target_coords <- tribble(
  ~gene,   ~chr, ~start,     ~end,
  "KDM6A", "X",  44873188,   45112779,
  "KDM6B", "17", 7834217,    7854796,
  "NSD1",  "5",  177131787,  177300213,
  "NSD2",  "4",  1871393,    1982192,
  "SETD2", "3",  47016436,   47164840
)

target_coords

In [ ]:
# Set helper functions

pick_col <- function(df, candidates, label) {
  hit <- candidates[candidates %in% names(df)][1]
  if (is.na(hit)) {
    stop(
      "Could not find a ", label, " column.\nAvailable columns are:\n",
      paste(names(df), collapse = ", ")
    )
  }
  hit
}

collapse_unique <- function(x) {
  x <- unique(na.omit(x))
  if (length(x) == 0) NA_character_ else paste(sort(x), collapse = ";")
}

tcga16 <- function(x) {
  str_sub(as.character(x), 1, 16)
}

patient_id <- function(x) {
  str_sub(as.character(x), 1, 12)
}

clean_chr <- function(x) {
  str_remove(as.character(x), "^chr")
}

In [1]:
# Test run on one TCGA project

process_project <- function(project) {
  message("Processing ", project)

  # Loss-of-function mutation calls
  maf <- tryCatch({
    q <- GDCquery(
      project = project,
      data.category = "Simple Nucleotide Variation",
      data.type = "Masked Somatic Mutation",
      workflow.type = "Aliquot Ensemble Somatic Variant Merging and Masking"
    )
    GDCdownload(q, directory = gdc_dir)
    GDCprepare(q, directory = gdc_dir)
  }, error = function(e) {
    message("  Mutation data unavailable: ", e$message)
    tibble()
  })

  lof_samples <- tibble()

  if (nrow(maf) > 0) {
    gene_col <- pick_col(maf, c("Hugo_Symbol", "gene", "Gene"), "gene")
    class_col <- pick_col(maf, c("Variant_Classification", "variant_classification"), "variant classification")
    barcode_col <- pick_col(maf, c("Tumor_Sample_Barcode", "sample", "Sample"), "tumour sample barcode")

    lof_samples <- maf %>%
      transmute(
        project = project,
        gene = .data[[gene_col]],
        mutation_class = .data[[class_col]],
        sample_barcode = .data[[barcode_col]],
        sample_id_16 = tcga16(sample_barcode),
        patient_id = patient_id(sample_barcode),
        event = "loss_of_function_mutation"
      ) %>%
      filter(gene %in% target_genes, mutation_class %in% lof_classes)
  }

  # Candidate homozygous deletion calls from copy-number segments
  cnv <- tryCatch({
    q <- GDCquery(
      project = project,
      data.category = "Copy Number Variation",
      data.type = "Masked Copy Number Segment"
    )
    GDCdownload(q, directory = gdc_dir)
    GDCprepare(q, directory = gdc_dir)
  }, error = function(e) {
    message("  Copy-number data unavailable: ", e$message)
    tibble()
  })

  homdel_samples <- tibble()

  if (nrow(cnv) > 0) {
    sample_col <- pick_col(cnv, c("Sample", "sample", "Tumor_Sample_Barcode", "GDC_Aliquot"), "copy-number sample")
    chr_col <- pick_col(cnv, c("Chromosome", "chromosome", "chrom", "chr"), "chromosome")
    start_col <- pick_col(cnv, c("Start", "Start_Position", "loc.start", "start"), "segment start")
    end_col <- pick_col(cnv, c("End", "End_Position", "loc.end", "end"), "segment end")
    seg_col <- pick_col(cnv, c("Segment_Mean", "SegmentMean", "seg.mean"), "segment mean")

    cnv_std <- cnv %>%
      transmute(
        project = project,
        sample_barcode = .data[[sample_col]],
        sample_id_16 = tcga16(sample_barcode),
        patient_id = patient_id(sample_barcode),
        chr = clean_chr(.data[[chr_col]]),
        segment_start = as.numeric(.data[[start_col]]),
        segment_end = as.numeric(.data[[end_col]]),
        segment_mean = as.numeric(.data[[seg_col]])
      )

    homdel_samples <- cnv_std %>%
      inner_join(target_coords, by = "chr") %>%
      filter(
        segment_start <= end,
        segment_end >= start,
        segment_mean <= homdel_threshold
      ) %>%
      transmute(
        project,
        gene,
        sample_id_16,
        patient_id,
        event = "candidate_homozygous_deletion",
        segment_mean,
        segment_start,
        segment_end
      )
  }

  events_long <- bind_rows(
    lof_samples %>%
      transmute(project, gene, sample_id_16, patient_id, event, mutation_class, segment_mean = NA_real_),
    homdel_samples %>%
      transmute(project, gene, sample_id_16, patient_id, event, mutation_class = NA_character_, segment_mean)
  )

  altered_samples <- events_long %>%
    group_by(project, sample_id_16, patient_id) %>%
    summarise(
      tumour_type = str_remove(project, "^TCGA-"),
      genes_lost = collapse_unique(gene),
      events = collapse_unique(event),
      lof_genes = collapse_unique(gene[event == "loss_of_function_mutation"]),
      homdel_genes = collapse_unique(gene[event == "candidate_homozygous_deletion"]),
      mutation_classes = collapse_unique(mutation_class),
      n_genes_lost = n_distinct(gene),
      n_loss_events = n(),
      has_lof = any(event == "loss_of_function_mutation"),
      has_homdel = any(event == "candidate_homozygous_deletion"),
      .groups = "drop"
    )

  summary <- tibble(
    project = project,
    tumour_type = str_remove(project, "^TCGA-"),
    n_lof_events = nrow(lof_samples),
    n_homdel_events = nrow(homdel_samples),
    n_altered_samples = nrow(altered_samples)
  )

  list(
    events_long = events_long,
    altered_samples = altered_samples,
    summary = summary
  )
}

In [ ]:
# Run cohort construction

project_results <- map(projects_to_run, process_project)

events_long <- map_dfr(project_results, "events_long")
altered_samples <- map_dfr(project_results, "altered_samples")
project_summary <- map_dfr(project_results, "summary")

write_csv(
  events_long,
  file.path(output_dir, paste0(gene_set_label, "_LoF_HomDel_events_long.csv"))
)

write_csv(
  altered_samples,
  file.path(output_dir, paste0(gene_set_label, "_altered_samples.csv"))
)

write_csv(
  project_summary,
  file.path(output_dir, paste0(gene_set_label, "_project_summary.csv"))
)

project_summary

In [ ]:
# Summarise altered samples

if (nrow(altered_samples) > 0) {
  gene_summary <- altered_samples %>%
    separate_rows(genes_lost, sep = ";") %>%
    count(genes_lost, name = "n_altered_samples") %>%
    rename(gene = genes_lost) %>%
    arrange(desc(n_altered_samples))

  tumour_summary <- altered_samples %>%
    count(tumour_type, name = "n_altered_samples") %>%
    arrange(desc(n_altered_samples))

  write_csv(
    gene_summary,
    file.path(output_dir, paste0(gene_set_label, "_gene_summary.csv"))
  )

  write_csv(
    tumour_summary,
    file.path(output_dir, paste0(gene_set_label, "_tumour_type_summary.csv"))
  )

  gene_summary
} else {
  message("No altered samples identified in the selected projects.")
}

In [ ]:
Save ouputs